# Module 3.4: Belief Revision

Facts change. People move cities, switch hotel chains, update dietary preferences.
What happens when an agent's memory says "Sarah lives in New York" but she moved
to Paris three months ago?

The obvious approaches — overwrite or append — both fail. This notebook demonstrates
why, then builds a **bi-temporal belief ledger** that handles changing facts correctly.

> **The question**: How should an agent handle facts that change over time?

In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
import sys, os, json
import sniffio
from datetime import datetime, timezone, timedelta

sys.path.insert(0, "..")
sniffio.current_async_library_cvar.set("asyncio")

from agent_framework._types import Message
from lifecycle_utils import (
    Triple, BiTemporalRecord, MemoryState
)
from shared.travel_agent import create_client

client, credential = create_client("../.env")
print("Setup complete")

## The Problem: Naive Approaches to Changing Facts

There are two obvious strategies for handling fact changes — both fail:

| Approach | What Happens | Problem |
|----------|-------------|---------|
| **Overwrite** (latest wins) | Old value deleted, new value stored | Can't audit. Can't answer "where did Sarah live in January?" |
| **Append-only** | Both values coexist in memory | Contradictions confuse retrieval. Agent says "you live in NYC AND Paris" |

Let's demonstrate both failures:

In [ ]:
# === Approach 1: Overwrite (latest wins) ===
overwrite_store = {"Sarah.lives_in": "New York"}  # January
print("January: Sarah lives in New York")
print(f"  Store: {overwrite_store}\n")

# March: Sarah moves to Paris — overwrite
overwrite_store["Sarah.lives_in"] = "Paris"
print("March: Sarah moves to Paris (overwrite)")
print(f"  Store: {overwrite_store}\n")

# Now try to answer: "Where did Sarah live in February?"
print("Query: 'Where did Sarah live in February 2025?'")
print(f"  Answer: {overwrite_store['Sarah.lives_in']}")  # WRONG! Says Paris
print("  ❌ WRONG — she lived in NYC in February, but we overwrote that fact\n")

print("=" * 60)

# === Approach 2: Append-only ===
append_store = [
    {"fact": "Sarah lives in New York", "stored": "2025-01-15"},
    {"fact": "Sarah lives in Paris", "stored": "2025-03-01"},
]
print("\n=== Approach 2: Append-only ===")
print("Store contains BOTH facts:")
for f in append_store:
    print(f"  • {f['fact']} (stored {f['stored']})")

print("\nQuery: 'Where does Sarah live?'")
print("  Retrieved: 'New York' AND 'Paris' — CONTRADICTION")
print("  ❌ Agent doesn't know which is current and which is historical")
print("\n→ Neither approach works. We need temporal tracking.")

## What Went Wrong

Both naive strategies lose information:
- **Overwrite** destroys history — can't audit, can't time-travel, can't explain
- **Append-only** creates contradictions — retrieval returns stale AND current values

The root cause: neither approach tracks **when** a fact was true in the real world.

## The Solution: Bi-Temporal Belief Tracking

Each fact gets TWO time dimensions:
- **Valid-time**: When was this true in the real world?
- **Transaction-time**: When did we store/learn this?

When a new fact supersedes an old one, the old fact's `valid_to` is set —
it's *retired*, not deleted. This lets us answer both:
- "Where does Sarah live NOW?" → Paris
- "Where did Sarah live in February?" → New York

> *"Temporal Validity in Retrieval Memory"* (arXiv:2606.26511) — shows that cosine
> similarity CANNOT distinguish contradicted facts from duplicates (AUROC 0.59).
> RAG serves superseded values 15-40% of the time. Bi-temporal tracking drives
> stale-fact errors to ~0%.

In [ ]:
class BiTemporalMemoryStore:
    """In-memory bi-temporal belief store for demonstration.
    
    In production, this would be backed by Cosmos DB with the
    'belief-ledger' container (partition key: /subject).
    """

    def __init__(self):
        self.records: list[BiTemporalRecord] = []

    def store(self, triple: Triple, valid_from: datetime = None,
              confidence: float = 0.8, source: str = "user_assertion") -> BiTemporalRecord:
        """Store a new belief, superseding any existing conflicting belief."""
        valid_from = valid_from or datetime.now(timezone.utc)

        # Check for supersession: same (subject, relation) with different object
        existing = self._find_current(triple.subject, triple.relation)
        if existing and existing.triple.object != triple.object:
            # Supersede the old belief
            new_id = None  # Will set after creating new record
            existing.supersede(new_record_id="pending", valid_to=valid_from)

        # Create new record
        record = BiTemporalRecord(
            triple=triple,
            valid_from=valid_from,
            confidence=confidence,
            source=source,
        )

        # Update supersession link
        if existing and existing.triple.object != triple.object:
            existing.superseded_by = record.id

        self.records.append(record)
        return record

    def query_current(self, subject: str, relation: str = None) -> list[BiTemporalRecord]:
        """Get currently valid beliefs for a subject."""
        results = []
        for r in self.records:
            if r.triple.subject == subject and r.is_current:
                if relation is None or r.triple.relation == relation:
                    results.append(r)
        return results

    def query_at_time(self, subject: str, at: datetime,
                      relation: str = None) -> list[BiTemporalRecord]:
        """Get beliefs that were valid at a specific point in time."""
        results = []
        for r in self.records:
            if r.triple.subject != subject:
                continue
            if relation and r.triple.relation != relation:
                continue
            # Check if this record was valid at the given time
            if r.valid_from <= at:
                if r.valid_to is None or r.valid_to > at:
                    results.append(r)
        return results

    def get_history(self, subject: str, relation: str) -> list[BiTemporalRecord]:
        """Get full evolution history for a (subject, relation) pair."""
        results = [r for r in self.records
                   if r.triple.subject == subject and r.triple.relation == relation]
        results.sort(key=lambda r: r.valid_from)
        return results

    def _find_current(self, subject: str, relation: str) -> BiTemporalRecord | None:
        """Find the currently valid record for (subject, relation)."""
        for r in self.records:
            if (r.triple.subject == subject and
                r.triple.relation == relation and r.is_current):
                return r
        return None


store = BiTemporalMemoryStore()
print("BiTemporalMemoryStore ready")

## The Payoff: Sarah Moves Cities

Let's run the **same scenario** that broke both naive approaches. This time,
the bi-temporal store handles it correctly:
- January: Sarah lives in New York
- March: Sarah moves to Paris
- Current query → Paris ✅
- Historical query ("February?") → New York ✅

In [ ]:
# January: Store initial fact
jan = datetime(2025, 1, 15, tzinfo=timezone.utc)
store.store(
    Triple(subject="Sarah", relation="lives_in", object="New York"),
    valid_from=jan,
    source="user_assertion",
)
print(f"January 15: Stored 'Sarah lives in New York'")

# March: Sarah moves — this supersedes the NYC fact
mar = datetime(2025, 3, 1, tzinfo=timezone.utc)
store.store(
    Triple(subject="Sarah", relation="lives_in", object="Paris"),
    valid_from=mar,
    source="user_assertion",
)
print(f"March 1:    Stored 'Sarah lives in Paris' (supersedes NYC)")
print()

In [ ]:
# Current query: Where does Sarah live NOW?
current = store.query_current("Sarah", "lives_in")
print("=== Current Query: 'Where does Sarah live?' ===")
for r in current:
    print(f"  → {r.triple.object} (since {r.valid_from.date()}, confidence={r.confidence})")

# Historical query: Where did Sarah live in February?
feb = datetime(2025, 2, 15, tzinfo=timezone.utc)
historical = store.query_at_time("Sarah", at=feb, relation="lives_in")
print(f"\n=== Historical Query: 'Where did Sarah live in February 2025?' ===")
for r in historical:
    print(f"  → {r.triple.object} (valid {r.valid_from.date()} to {r.valid_to.date() if r.valid_to else 'present'})")

In [ ]:
# Full evolution history
history = store.get_history("Sarah", "lives_in")
print("=== Belief Evolution: Sarah.lives_in ===")
print()
for i, r in enumerate(history):
    status = "✅ CURRENT" if r.is_current else "⏹ SUPERSEDED"
    valid_range = f"{r.valid_from.date()} → {r.valid_to.date() if r.valid_to else 'present'}"
    print(f"  [{i+1}] {r.triple.object:<15} | {valid_range} | {status}")
    if r.superseded_by:
        print(f"      └─ superseded by: {r.superseded_by[:8]}...")
    print()

## Edge Case: Visiting vs Moving

Not every location mention is a permanent change. The agent needs to distinguish:
- "I just moved to Paris" → permanent (supersedes)
- "I'm visiting Paris next week" → temporary (does NOT supersede)

We use LLM classification to determine if a new statement represents a
permanent change or a temporary state.

In [ ]:
REVISION_PROMPT = """You are a belief revision classifier. Given an existing belief and a new statement,
determine if the new statement SUPERSEDES the existing belief.

Existing belief: {existing}
New statement: {new_statement}

Respond with ONLY valid JSON:
{{
  "supersedes": true/false,
  "new_fact": "extracted fact if supersedes, else null",
  "change_type": "permanent|temporary|unrelated",
  "confidence": 0.0-1.0,
  "reasoning": "brief explanation"
}}"""


async def classify_revision(existing_belief: str, new_statement: str) -> dict:
    """Use LLM to determine if new statement supersedes existing belief."""
    prompt = REVISION_PROMPT.format(
        existing=existing_belief,
        new_statement=new_statement,
    )
    messages = [
        Message(role="system", contents=[prompt]),
        Message(role="user", contents=["Classify this revision."]),
    ]
    response = await client.get_response(messages=messages)
    raw = response.text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(raw)

print("Revision classifier ready")

In [ ]:
# Test ambiguous cases
test_cases = [
    ("Sarah lives in New York", "I just moved to Paris for my new job"),
    ("Sarah lives in New York", "I'm visiting Paris next week for a conference"),
    ("Sarah prefers Marriott", "The Hilton was actually amazing this time"),
    ("Sarah prefers Marriott", "I've switched to Hilton — their loyalty program is better"),
]

print("=== Revision Classification ===\n")
for existing, new in test_cases:
    result = await classify_revision(existing, new)
    icon = "🔄" if result["supersedes"] else "➡️"
    print(f"{icon} Existing: \"{existing}\"")
    print(f"   New:      \"{new}\"")
    print(f"   Supersedes: {result['supersedes']} ({result['change_type']})")
    print(f"   Confidence: {result['confidence']}")
    print(f"   Reasoning:  {result['reasoning']}")
    print()

## Scaling Up: A Full Belief Profile

Let's populate a complete belief store for Sarah over 15 months and demonstrate
current vs historical queries across multiple facts.

In [ ]:
# Build Sarah's belief profile over time
store2 = BiTemporalMemoryStore()

# 2024-01: Initial facts
t0 = datetime(2024, 1, 1, tzinfo=timezone.utc)
store2.store(Triple("Sarah", "lives_in", "New York"), valid_from=t0)
store2.store(Triple("Sarah", "prefers_hotel", "Marriott"), valid_from=t0)
store2.store(Triple("Sarah", "prefers_seat", "window"), valid_from=t0)
store2.store(Triple("Sarah", "home_airport", "JFK"), valid_from=t0)
store2.store(Triple("Sarah", "diet", "vegetarian"), valid_from=t0)

# 2024-06: Some facts evolve
t1 = datetime(2024, 6, 1, tzinfo=timezone.utc)
store2.store(Triple("Sarah", "prefers_hotel", "Hilton"), valid_from=t1)  # Changed!

# 2025-03: Moves to Paris
t2 = datetime(2025, 3, 1, tzinfo=timezone.utc)
store2.store(Triple("Sarah", "lives_in", "Paris"), valid_from=t2)
store2.store(Triple("Sarah", "home_airport", "CDG"), valid_from=t2)

print("=== Sarah's Current Beliefs ===")
current_beliefs = store2.query_current("Sarah")
for r in sorted(current_beliefs, key=lambda x: x.triple.relation):
    print(f"  {r.triple.relation:<20} = {r.triple.object:<15} (since {r.valid_from.date()})")

print(f"\n=== Sarah's Beliefs in April 2024 ===")
apr_2024 = datetime(2024, 4, 1, tzinfo=timezone.utc)
old_beliefs = store2.query_at_time("Sarah", at=apr_2024)
for r in sorted(old_beliefs, key=lambda x: x.triple.relation):
    print(f"  {r.triple.relation:<20} = {r.triple.object:<15} (since {r.valid_from.date()})")

## Audit Trail: Belief Timelines

For governance and debugging, the bi-temporal store provides a complete
audit trail of how any belief evolved over time.

In [ ]:
def print_belief_timeline(store: BiTemporalMemoryStore, subject: str, relation: str):
    """Print a visual timeline of belief evolution."""
    history = store.get_history(subject, relation)
    if not history:
        print(f"  No history for ({subject}, {relation})")
        return

    print(f"  Timeline for: {subject}.{relation}")
    print(f"  {'─' * 50}")
    for r in history:
        start = r.valid_from.strftime("%Y-%m-%d")
        end = r.valid_to.strftime("%Y-%m-%d") if r.valid_to else "present   "
        state_icon = "✅" if r.is_current else "⏹"
        print(f"  {state_icon} {start} → {end}  │ {r.triple.object}")
    print()

print("=== Belief Timelines ===")
print()
print_belief_timeline(store2, "Sarah", "lives_in")
print_belief_timeline(store2, "Sarah", "prefers_hotel")
print_belief_timeline(store2, "Sarah", "home_airport")

## Production: Cosmos DB Serialization

In production, the belief ledger is stored in a Cosmos DB container with
partition key `/subject`. Here's what the documents look like.

In [ ]:
# Show what Cosmos documents look like
sample_records = store2.get_history("Sarah", "lives_in")
print("=== Cosmos DB Documents for 'Sarah.lives_in' ===")
print()
for r in sample_records:
    doc = r.to_cosmos()
    print(json.dumps(doc, indent=2))
    print()

## Key Takeaways

1. **Never just overwrite** — use supersession with temporal validity
2. **Two time dimensions** — valid-time (world truth) vs transaction-time (when stored)
3. **Supersession detection** — same (subject, relation) with different object triggers revision
4. **Ambiguity resolution** — LLM classifies if a change is permanent or temporary
5. **Time-travel queries** — answer "what did we believe at time T?" for any T
6. **Full audit trail** — every belief change is traceable with provenance

## Next: Retention & Decay (Notebook 05)

The belief store grows over time. The next notebook implements bounded memory
with scoring-based retention — ensuring old, unused, or redundant beliefs are
gracefully evicted or compressed rather than accumulated indefinitely.